In [ ]:
from tqdm.keras import TqdmCallback
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import KFold
from tqdm import tqdm
import optuna



def build_model(input_shape, learning_rate):
    model = models.Sequential([
        layers.Input(shape=(input_shape,)),
        layers.Dense(1024, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(512, activation='relu'),
        layers.Dense(64, activation='relu'),
        layers.Dense(1),
    ])
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error')
    return model

def objective(trial):
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical('batch_size', [32, 64, 100, 128])

    kfold = KFold(n_splits=3, shuffle=True, random_state=42)
    cv_losses = []

    for train_idx, val_idx in kfold.split(X):
        X_train_cv, X_val_cv = X[train_idx], X[val_idx]
        y_train_cv, y_val_cv = y[train_idx], y[val_idx]

        model = build_model(X.shape[1], learning_rate)

        es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

        model.fit(
            X_train_cv, y_train_cv,
            validation_data=(X_val_cv, y_val_cv),
            epochs=50,
            batch_size=batch_size,
            verbose=0,
            callbacks=[es, TqdmCallback(verbose=0)]  # تغییر داده شده
        )

        val_loss = model.evaluate(X_val_cv, y_val_cv, verbose=0)
        cv_losses.append(val_loss)

    return np.mean(cv_losses)

N_TRIALS = 10
with tqdm(total=N_TRIALS) as pbar:
    def tqdm_callback(study, trial):
        pbar.update(1)

    study = optuna.create_study(direction='minimize')
    study.optimize(
        objective,
        n_trials=10,
        callbacks=[tqdm_callback]  # توجه: اینجا باید callback خودتان باشد نه TqdmCallback
    )

print(" Best parameters:", study.best_trial.params)
print(" Best CV loss:", study.best_trial.value)

best_lr = study.best_trial.params['learning_rate']
best_bs = study.best_trial.params['batch_size']

final_model = build_model(X.shape[1], best_lr)
es = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = final_model.fit(
    X, y,
    validation_split=0.2,
    epochs=100,
    batch_size=best_bs,
    callbacks=[es, TqdmCallback()],
    verbose=1
)

final_model.save("best_dnn_model.keras")

plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training vs Validation Loss')
plt.legend()
plt.grid()
plt.savefig("loss_plot.png")
plt.show()